### 2-1

In [8]:
import pandas as pd

In [7]:
df1 = pd.read_csv("../../raw/34/data/m2_source.csv")


# display(df1.head(6))

import numpy as np

print("결측", df1.isna().sum().sum())


df1_val = df1.iloc[:, 2:]
df1_val = df1_val.transpose()



temp1 = df1_val.interpolate(method='linear', limit_direction='both')
# temp2 = (df1.fillna(method='bfill').fillna(method='ffill') + df1.fillna(method='ffill').fillna(method='bfill'))/2

df1.iloc[:, 2:] = temp1.transpose()

# ========================================================

df1 = df1.groupby(by=["일자"]).mean()*2

# df1['season']

df1 = df1.reset_index()

# df1['일자'].dt.month

df1['season'] = df1['일자'].apply(lambda x: x[5:7]).astype(int)

df1['season'] = df1['season'].map({3: 0, 4: 0, 5: 0, 6: 1, 7: 1, 8: 1, 9: 2, 10: 2, 11: 3, 12: 3})

df1 = df1.dropna()

# df1


df1 = df1.melt(id_vars=['일자', 'season'], var_name='hour', value_name='전력량')




df1['hour'] = (df1['hour'].str.replace('hour_', '').astype(int)-1).astype(str).apply(lambda x: x.rjust(2,'0'))


df1['timestamp'] = pd.to_datetime(df1['일자'] + ' ' + df1['hour']+':00:00')

# df1.info()
df1.head()


결측 242


,일자,season,hour,전력량,timestamp
0,2021-03-01,0.0,00,99.59,2021-03-01
1,2021-03-02,0.0,00,162.78,2021-03-02
2,2021-03-03,0.0,00,17.13,2021-03-03
3,2021-03-04,0.0,00,10.87,2021-03-04
4,2021-03-05,0.0,00,111.54,2021-03-05


In [9]:
df2 = pd.read_csv("../../raw/34/data/m2_weather.csv")

df2['일시'] = pd.to_datetime(df2['일시'].str.split("@").apply(lambda x: x[0]).str.replace("_", "-")
+ " "
+ df2['일시'].str.split("@").apply(lambda x: x[1]).apply(lambda x: x.rjust(2, "0"))
+ ":00:00")


df2[['강수량', '일조', '일사', '적설']] = df2[['강수량', '일조', '일사', '적설']].fillna(value=0)

df2[['풍속', '습도']] = df2[['풍속', '습도']].fillna(method='ffill')

df2[['전운량', '지면온도']] = df2[['전운량', '지면온도']].fillna(df2[['전운량', '지면온도']].mean())

In [11]:
df0 = pd.merge(df1, df2, left_on='timestamp', right_on='일시')[['일시', '전력량', 'season', '기온', '강수량', '풍속', '습도', '일조', '일사', '적설', '전운량', '지면온도']]

df0 = df0.rename(columns={'전력량': '총전력량'})

df0.head(2)
# help(pd.merge)


,일시,총전력량,season,기온,강수량,풍속,습도,일조,일사,적설,전운량,지면온도
0,2021-03-01,99.59,0.0,16.6,0.0,4.6,73.0,0.0,0.0,0.0,6.0,12.3
1,2021-03-02,162.78,0.0,8.2,2.4,6.1,85.0,0.0,0.0,0.0,10.0,8.3


In [14]:
import pandas as pd
import pandas_profiling
import matplotlib.pyplot as plt

# 1. matplotlib 폰트 및 마이너스 기호 설정 (리포트 생성 전 필수)
plt.rcParams['font.family'] = 'Malgun Gothic'  # 윈도우 기준 '맑은 고딕'
plt.rcParams['axes.unicode_minus'] = False

# 충돌을 유발하는 phi_k와 cramers 연산만 명시적으로 False 처리
profile = df0.profile_report(
    title='Pandas Profiling Report',
    correlations={
        "pearson": True,
        "spearman": True,
        "kendall": True,
        "phi_k": False,
        "cramers": False
    }
)

profile
# profile.to_notebook_iframe()